In [3]:
# Import necessary packages
import pandas as pd
import numpy as np

# Read in the data
df = pd.read_csv('../data/raw_lung_cancer_data.csv')
df

,year_of_diagnosis,dx_lastcontact_death_months,puf_vital_status,puf_case_id,puf_facility_id,facility_type_cd,facility_location_cd,age,sex,race,...,mets_at_dx_liver,mets_at_dx_lung,mets_at_dx_other,tumor_size_summary_2016,rx_summ_surg_prim_site,rx_summ_treatment_status,rx_summ_chemo,rx_summ_immunotherapy,rx_summ_surgrad_seq,tumor_size_na
0,2018,1.120000,Dead,D00002e7d-bb34-46e6-849e-6c4f0acef147,RNYCFBVPLK,Community Cancer Program,South Atlantic,71,Male,White,...,NaN,Yes,Yes,94.0,NaN,No Treatment Given,NaN,NaN,No radiation therapy and/or surgical procedures,False
1,2019,34.070000,Dead,D00002f04-3dd7-4234-bb33-84f3a20b86c9,PSAHKJHEIG,Comprehensive Community Cancer Program,Pacific,Ninety or older,Female,Black,...,NaN,Yes,NaN,28.0,NaN,Treatment Given,NaN,"Immunotherapy recommended, unknown if administ...",No radiation therapy and/or surgical procedures,False
2,2019,21.450001,Dead,D0000690b-4418-4162-bc8e-2d62701c93ca,GQSMCSUAOK,Community Cancer Program,East North Central,80,Female,White,...,NaN,NaN,NaN,12.0,NaN,Treatment Given,Multiagent chemotherapy,NaN,No radiation therapy and/or surgical procedures,False
3,2022,1.380000,Dead,D00006b9c-dcb3-4535-a4fd-e20bd886fc9d,AUUCOIRDTB,Comprehensive Community Cancer Program,South Atlantic,80,Female,American Indian,...,NaN,NaN,Yes,87.0,NaN,Treatment Given,NaN,Immunotherapy administered as first course the...,No radiation therapy and/or surgical procedures,False
4,2019,40.049999,Alive,D00006dce-2eb8-498e-a6bc-da673b7a434e,CFHJPZWJNY,Integrated Network Cancer Program,West South Central,57,Female,White,...,NaN,NaN,NaN,NaN,NaN,Treatment Given,Multiagent chemotherapy,Immunotherapy administered as first course the...,No radiation therapy and/or surgical procedures,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501364,2020,7.360000,Dead,Dffff7a97-fd34-438d-b5b9-ca1930c78679,TXEIDOZHLI,Integrated Network Cancer Program,South Atlantic,75,Male,White,...,NaN,NaN,NaN,103.0,56.0,Treatment Given,Multiagent chemotherapy,Immunotherapy administered as first course the...,No radiation therapy and/or surgical procedures,False
501365,2018,12.160000,Dead,Dffff8d6c-a1cd-4085-9846-222b943a44d3,TVXWQGNICT,Comprehensive Community Cancer Program,Pacific,61,Female,White,...,NaN,NaN,NaN,46.0,NaN,Treatment Given,Multiagent chemotherapy,Immunotherapy administered as first course the...,No radiation therapy and/or surgical procedures,False
501366,2022,1.870000,Dead,Dffffabfc-6747-43e7-86c7-9cf043363c9f,UMUHJVXWPB,Integrated Network Cancer Program,West South Central,75,Male,Other,...,NaN,Yes,NaN,NaN,NaN,No Treatment Given,NaN,NaN,No radiation therapy and/or surgical procedures,True
501367,2018,38.410000,Alive,Dffffac4c-3b09-40a9-bfb0-45bd6333893d,JDTRSDGOHO,Community Cancer Program,West South Central,70,Male,White,...,Yes,NaN,NaN,32.0,12.0,Treatment Given,NaN,NaN,Radiation therapy after surgery,False


In [4]:
# Perform some cleaning

# Convert "Ninety or older" to 90 and convert to numeric
df['age'] = df['age'].replace('Ninety or older', 90).astype(float)

# Reclassify small/large tumor sizes and convert to numeric
df['tumor_size_summary_2016'] = df['tumor_size_summary_2016'].replace('No Tumor found', 0)
df['tumor_size_summary_2016'] = df['tumor_size_summary_2016'].replace('1 mm or < 1 mm', 1)
df['tumor_size_summary_2016'] = df['tumor_size_summary_2016'].replace('>=989 mm', 989)
df['tumor_size_summary_2016'] = df['tumor_size_summary_2016'].replace('Site Specific Codes', np.nan)
df['tumor_size_summary_2016'] = df['tumor_size_summary_2016'].replace(
    'Microscopic focus/foci only/no size of focus', 0).astype(float)

# Impute missing values for 'tumor_size_summary_2016'
np.random.seed(492026)
missing = df['tumor_size_summary_2016'].isna()
n_missing = missing.sum()
observed_values = df.loc[~missing, 'tumor_size_summary_2016']
imputed_values = np.random.choice(observed_values, size=n_missing, replace=True)
df.loc[missing, 'tumor_size_summary_2016'] = imputed_values

# Set any missing values in categorical columns to 'Unknown'/'None'
df['ur_cd_23'] = df['ur_cd_23'].fillna('Unknown')
df['mets_at_dx_bone'] = df['mets_at_dx_bone'].fillna('None')
df['mets_at_dx_brain'] = df['mets_at_dx_brain'].fillna('None')
df['mets_at_dx_distant_ln'] = df['mets_at_dx_distant_ln'].fillna('None')
df['mets_at_dx_liver'] = df['mets_at_dx_liver'].fillna('None')
df['mets_at_dx_lung'] = df['mets_at_dx_lung'].fillna('None')
df['mets_at_dx_other'] = df['mets_at_dx_other'].fillna('None')
df['rx_summ_surg_prim_site'] = df['rx_summ_surg_prim_site'].fillna('None')
df['rx_summ_chemo'] = df['rx_summ_chemo'].fillna('None')
df['rx_summ_immunotherapy'] = df['rx_summ_immunotherapy'].fillna('None')

# Drop "lymph_vascular_invasion" due to >50% missingness
df = df.drop(columns=['lymph_vascular_invasion'])

# Drop "tobacco_use" due to >80% missingness
df = df.drop(columns=['tobacco_use'])

# Drop "rx_summ_treatment_status"
df = df.drop(columns=['rx_summ_treatment_status'])

# Drop "tumor_size_na"
df = df.drop(columns=['tumor_size_na'])

# Drop "Other specified types of cancer programs"
df = df[df['facility_type_cd'] != 'Other specified types of cancer programs']

# Combine 'race' categories
race_dict = {
    'White': 'White',
    'Black': 'Black',
    'Asian': 'Asian',
    'Other': 'Other',
    'American Indian': 'Other',
    'Asian Indian': 'Asian',
    'Pacific Islander': 'Other'
}
df['race'] = df['race'].replace(race_dict)

# Drop observations with certain "laterality" values
df = df[df['laterality'] != 'Organ is not considered to be a paired site']
df = df[df['laterality'] != 'Only one side involved, right or left origin not specified']
df = df[df['laterality'] != 'Paired site midline tumor']

# Condense "histology" categories
histology_map = {
    # --- Adenocarcinoma & variants ---
    8200: "Adenocarcinoma", 8201: "Adenocarcinoma",
    8210: "Adenocarcinoma",
    8250: "Adenocarcinoma", 8251: "Adenocarcinoma", 8252: "Adenocarcinoma",
    8253: "Adenocarcinoma", 8254: "Adenocarcinoma", 8255: "Adenocarcinoma",
    8256: "Adenocarcinoma", 8257: "Adenocarcinoma",
    8260: "Adenocarcinoma", 8265: "Adenocarcinoma",
    8310: "Adenocarcinoma",
    8320: "Adenocarcinoma", 8323: "Adenocarcinoma",
    8333: "Adenocarcinoma",
    8410: "Adenocarcinoma",
    8430: "Adenocarcinoma",
    8480: "Adenocarcinoma", 8481: "Adenocarcinoma",
    8490: "Adenocarcinoma",
    8500: "Adenocarcinoma", 8503: "Adenocarcinoma",
    8524: "Adenocarcinoma",
    8551: "Adenocarcinoma",
    8560: "Adenocarcinoma", 8562: "Adenocarcinoma",
    8570: "Adenocarcinoma", 8574: "Adenocarcinoma",
    8575: "Adenocarcinoma", 8576: "Adenocarcinoma",
    8140: "Adenocarcinoma",
    8144: "Adenocarcinoma",
    8147: "Adenocarcinoma",
    # --- Undifferentiated / anaplastic carcinoma ---
    8020: "Undifferentiated carcinoma",
    8021: "Undifferentiated carcinoma",
    8022: "Undifferentiated carcinoma",
    8023: "Undifferentiated carcinoma",
    8030: "Undifferentiated carcinoma",
    8031: "Undifferentiated carcinoma",
    8032: "Undifferentiated carcinoma",
    8033: "Undifferentiated carcinoma",
    8035: "Undifferentiated carcinoma",
    # --- Rare epithelial ---
    8290: "Adenocarcinoma",
    8441: "Adenocarcinoma",
    # --- Squamous cell carcinoma ---
    **{code: "Squamous cell carcinoma" for code in range(8050, 8053)},
    **{code: "Squamous cell carcinoma" for code in range(8070, 8077)},
    **{code: "Squamous cell carcinoma" for code in range(8082, 8085)},
    8094: "Squamous cell carcinoma",
    # --- Neuroendocrine ---
    8013: "Neuroendocrine tumor",
    8230: "Neuroendocrine tumor",
    **{code: "Neuroendocrine tumor" for code in range(8240, 8250)},
    # --- Small cell ---
    8046: "Small cell carcinoma",
    # --- Undifferentiated / NOS ---
    8012: "Undifferentiated carcinoma",
    8014: "Undifferentiated carcinoma",
    8120: "Undifferentiated carcinoma",
    8123: "Undifferentiated carcinoma",
}
df['histology'] = df['histology'].map(histology_map)

# Condense "regional_nodes_positive" categories
df = df[
    df['regional_nodes_positive'] != 'Positive nodes are documented, but the number is unspecified'
    ]
string = 'Unknown whether nodes are positive, not applicable, not stated in patient record'
df = df[df['regional_nodes_positive'] != string]
string = 'Positive nodes are documented, but the number is unspecified'
df = df[df['regional_nodes_positive'] != string]
node_map = {
    'No nodes were examined':'No nodes were examined',
    'All nodes examined are negative':'All nodes examined are negative',
    'Positive aspiration of lymph node(s) was performed':'Positive aspiration of lymph node(s)',
    **{count:"1-10" for count in [str(i) for i in range(1, 11)]},
    **{count:"11-20" for count in [str(i) for i in range(11, 21)]},
    **{count:"21+" for count in [str(i) for i in range(21, 90)]}
}
df['regional_nodes_positive'] = df['regional_nodes_positive'].map(node_map)

# Drop observations with certain values for "analytic_stage_group"
df = df[df['analytic_stage_group'] != 'AJCC Staging not applicable']
df = df[df['analytic_stage_group'] != 'Stage 0']
df = df[df['analytic_stage_group'] != 'Occult (lung only)']

# Combine "mets_at_dx_other" categories
df['mets_at_dx_other'] = df['mets_at_dx_other'].replace(
    {'Generalized metastases such as carcinomatosis':'Yes'}
    )

# Condense "rx_summ_surg_prim_site" categories
surg_map = {
    # No surgery
    "None": "No surgery",
    # Local destruction
    '12.0': "Local destruction",
    '13.0': "Local destruction",
    '15.0': "Local destruction",
    '19.0': "Local destruction",
    # Sublobar — split
    '21.0': "Wedge resection",
    '22.0': "Segmentectomy",
    '20.0': "Sublobar NOS",
    '23.0': "Sublobar NOS",
    '24.0': "Sublobar NOS",
    '25.0': "Sublobar NOS",
    # Lobectomy / bilobectomy
    '30.0': "Lobectomy",
    '33.0': "Lobectomy",
    '45.0': "Lobectomy",
    '46.0': "Lobectomy",
    '47.0': "Lobectomy",
    '48.0': "Lobectomy",
    # Pneumonectomy
    '55.0': "Pneumonectomy",
    '56.0': "Pneumonectomy",
    '65.0': "Pneumonectomy",
    '66.0': "Pneumonectomy",
    '70.0': "Pneumonectomy",
    # Surgery NOS
    '80.0': "Surgery, NOS",
    "Surgery, NOS": "Surgery, NOS",
    # No surgery
    'No surgery':'None'
}
df["rx_summ_surg_prim_site"] = df["rx_summ_surg_prim_site"].map(surg_map).fillna('None')

# Combine "rx_summ_chemo" values
df = df[
    df['rx_summ_chemo'] != 'Chemotherapy administered, type and number of agents not documented'
    ]
chemo_map = {
    "None":'None',
    "Multiagent chemotherapy":"Multiagent chemotherapy",
    "Single-agent chemotherapy":"Single-agent chemotherapy"
}
df['rx_summ_chemo'] = df['rx_summ_chemo'].map(chemo_map).fillna("None")

# Combine "rx_summ_immunotherapy" values
immuno_map = {
    "None":'None',
    "Immunotherapy administered as first course therapy":"Yes (first course therapy)",
}
df['rx_summ_immunotherapy'] = df['rx_summ_immunotherapy'].map(immuno_map).fillna("None")

# Drop observations with certain values for "rx_summ_surgrad_seq"
df = df[df['rx_summ_surgrad_seq'] != 'Intraoperative radiation therapy']
string = 'Intraoperative radiation therapy with other therapy administered before or after surgery'
df = df[df['rx_summ_surgrad_seq'] != string]

# Map "primary_site" values to words
site_map = {
    "C340": "Main bronchus",
    "C341": "Upper lobe, lung",
    "C342": "Middle lobe, lung",
    "C343": "Lower lobe, lung",
    "C348": "Overlapping lesion of lung",
    "C349": "Lung, NOS",
}
df["primary_site"] = df["primary_site"].map(site_map)

# Drop observations that have "Unknown" values for any column
df = df.reset_index(drop=True)
indices_to_drop = []
for i in range(df.shape[0]):
    for j in range(df.shape[1]):
        if "unknown" in str(df.iloc[i, j]).lower():
            indices_to_drop.append(i)
df = df.drop(index=indices_to_drop)
df = df.reset_index(drop=True)

In [6]:
# Save the cleaned data
df.to_csv('../data/lung_cancer_data.csv', index=False)